<a href="https://colab.research.google.com/github/Riddhima76/Airlines_Sentiment_Analysis/blob/main/Airlines_Sentiment_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [38]:
import pandas as pd    #for handling data
pd.set_option('display.max_colwidth',200)  #Increase the output column width
import numpy as np    #for numerical computing
import re        #for pattern matching
import spacy     #for NLP related tasks
nlp=spacy.load('en_core_web_sm',disable=['tagger','parser','ner'])

### Steps to Follow
1. Loading and Exploring Data
2. Text Cleaning
3. Data Preparation
    1. Label Encoding
    2. Split Data
    3. Feature Engineering using TF-IDF
4. Model Building
    1. Naive Bayes
    2. Logistic Regression
    3. Model Building Summary
5. Final Sentiment Analysis Pipeline

Loading and Exploring Data

In [2]:
df=pd.read_csv('Tweets_airlines.csv', encoding='utf-8')

In [3]:
df.shape

(14640, 15)

In [4]:
df.head()

,tweet_id,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,airline_sentiment_gold,name,negativereason_gold,retweet_count,text,tweet_coord,tweet_created,tweet_location,user_timezone
0,570306133677760513,neutral,1.0000,NaN,NaN,Virgin America,NaN,cairdin,NaN,0,@VirginAmerica What @dhepburn said.,NaN,2015-02-24 11:35:52 -0800,NaN,Eastern Time (US & Canada)
1,570301130888122368,positive,0.3486,NaN,0.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica plus you've added commercials to the experience... tacky.,NaN,2015-02-24 11:15:59 -0800,NaN,Pacific Time (US & Canada)
2,570301083672813571,neutral,0.6837,NaN,NaN,Virgin America,NaN,yvonnalynn,NaN,0,@VirginAmerica I didn't today... Must mean I need to take another trip!,NaN,2015-02-24 11:15:48 -0800,Lets Play,Central Time (US & Canada)
3,570301031407624196,negative,1.0000,Bad Flight,0.7033,Virgin America,NaN,jnardino,NaN,0,"@VirginAmerica it's really aggressive to blast obnoxious ""entertainment"" in your guests' faces &amp; they have little recourse",NaN,2015-02-24 11:15:36 -0800,NaN,Pacific Time (US & Canada)
4,570300817074462722,negative,1.0000,Can't Tell,1.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica and it's a really big bad thing about it,NaN,2015-02-24 11:14:45 -0800,NaN,Pacific Time (US & Canada)


In [5]:
df['text'].sample(5)

,text
12036,@AmericanAir Hopefully you ll see bad ones as opportunity to get better and not dwell in it... and the good ones as encouragement words!
14215,@AmericanAir Trip cut 7 hrs short due to flight change/massive layovers/ no peanuts or crackers/ asked for H2O and received about 2oz.
5172,@SouthwestAir how you gonna Cancelled Flight my flight but run flights at the exact same time? Cmon fam
14384,"@AmericanAir \n\nLate Flight from phoenix, gave ticks away when arrived chicago, held bags hostage chicago, rented van drove to toledo. 5 hr drive."
14051,@AmericanAir VERY upset that I cannot select seats for Tuesday flight online or over the phone. Terrible customer service :( Please help!


In [6]:
df['airline_sentiment'].value_counts()

,count
airline_sentiment,
negative,9178
neutral,3099
positive,2363


In [7]:
df['airline_sentiment'].value_counts(normalize=True)*100

,proportion
airline_sentiment,
negative,62.691257
neutral,21.168033
positive,16.140710


Text Cleaning

In [9]:
def text_cleaner(text):
  text=re.sub(r'@[A-Za-z0-9]+','',text)  #Remove user mentions
  text=re.sub(r'#[A-Za-z0-9]+','',text)  #Remove hashtags
  text=re.sub(r'http\S+','',text)    #Remove links
  text=text.lower()          #Lowercasing the text
  text=re.sub(r"[^a-z]+"," ",text)   #fetch only words
  text=re.sub(r"[\s]+"," ",text)     #Removing extra spaces
  doc=nlp(text)    #creating doc objects
  tokens=[token.lemma_ for token in doc if (token.is_stop==False)]   #Removing stopwords and lemmatizing the text
  return " ".join(tokens)   #Join tokens by space

In [10]:
df['clean_text']=df['text'].apply(text_cleaner)

/usr/local/lib/python3.13/dist-packages/spacy/pipeline/lemmatizer.py:187: UserWarning: [W108] The rule-based lemmatizer did not find POS annotation for one or more tokens. Check that your pipeline includes components that assign token.pos, typically 'tagger'+'attribute_ruler' or 'morphologizer'.
  warnings.warn(Warnings.W108)


In [11]:
text=df['clean_text'].values
labels=df['airline_sentiment'].values

In [12]:
text[:10]

array(['  said', '  plus ve added commercials experience tacky',
       '  didn t today mean need trip',
       '  s aggressive blast obnoxious entertainment guests faces amp little recourse',
       '  s big bad thing',
       '  seriously pay flight seats didn t playing s bad thing flying va',
       '  yes nearly time fly vx ear worm won t away',
       '  missed prime opportunity men hats parody', '  didn t d',
       '  amazing arrived hour early good'], dtype=object)

In [13]:
labels[:10]

array(['neutral', 'positive', 'neutral', 'negative', 'negative',
       'negative', 'positive', 'neutral', 'positive', 'positive'],
      dtype=object)

### Data Preparation

Label Encoding

In [14]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
labels=le.fit_transform(labels)

In [15]:
labels[:10]

array([1, 2, 1, 0, 0, 0, 2, 1, 2, 2])

In [16]:
le.inverse_transform([0,1,2])

array(['negative', 'neutral', 'positive'], dtype=object)

Split Data

In [17]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(text,labels,stratify=labels,test_size=0.2,random_state=0,shuffle=True)

In [18]:
print('x_train:',x_train.shape,'y_train:',y_train.shape)
print('x_test:',x_test.shape,'y_test:',y_test.shape)

x_train: (11712,) y_train: (11712,)
x_test: (2928,) y_test: (2928,)


Feature Engineering using TF-IDF

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer
word_vectorizer=TfidfVectorizer(max_features=1000)

In [20]:
word_vectorizer.fit(x_train)

TfidfVectorizer(max_features=1000)

In [21]:
train_word_features=word_vectorizer.transform(x_train)
train_word_features

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 64939 stored elements and shape (11712, 1000)>

In [22]:
test_word_features=word_vectorizer.transform(x_test)
test_word_features

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 16368 stored elements and shape (2928, 1000)>

### Model Building

Naive Bayes

In [25]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import f1_score
nb_model=MultinomialNB().fit(train_word_features,y_train)
nb_model

MultinomialNB()

In [26]:
train_pred_nb=nb_model.predict(train_word_features)
train_pred_nb

array([0, 0, 0, ..., 2, 0, 0])

In [27]:
print("F1-score on train set: ",f1_score(y_train,train_pred_nb,average='weighted'))

F1-score on train set:  0.7269836468528603


In [28]:
test_pred_nb=nb_model.predict(test_word_features)
print("F1-score on test set: ",f1_score(y_test,test_pred_nb,average='weighted'))

F1-score on test set:  0.6843831187586511


Logistic Regression

In [29]:
from sklearn.linear_model import LogisticRegression

In [31]:
lr_model=LogisticRegression().fit(train_word_features,y_train)
lr_model

LogisticRegression()

In [32]:
train_pred_lr=lr_model.predict(train_word_features)
train_pred_lr

array([2, 0, 1, ..., 2, 0, 0])

In [33]:
print("F1-score on Train set: ",f1_score(y_train,train_pred_lr,average='weighted'))

F1-score on Train set:  0.8055045023259025


In [35]:
test_pred_lr=lr_model.predict(test_word_features)
print("F1-score on Test set: ",f1_score(y_test,test_pred_lr,average='weighted'))

F1-score on Test set:  0.7469408762504411


Observation: Logistic regression performs better than Naive Bayes for this dataset

In [36]:
def sentiment_analyzer(tweet):
  cleaned_tweet=text_cleaner(tweet)
  tweet_vector=word_vectorizer.transform([cleaned_tweet])
  label=lr_model.predict(tweet_vector)
  return le.inverse_transform(np.array(label))

In [37]:
sentiment_analyzer("@USAirways flt 419. 2+ hrs Late Flight, baggage + 1 more hr. Now I see they delivered my suitcase wet inside & out. #NotHappy")

/usr/local/lib/python3.13/dist-packages/spacy/pipeline/lemmatizer.py:187: UserWarning: [W108] The rule-based lemmatizer did not find POS annotation for one or more tokens. Check that your pipeline includes components that assign token.pos, typically 'tagger'+'attribute_ruler' or 'morphologizer'.
  warnings.warn(Warnings.W108)


array(['negative'], dtype=object)